# LangSmith Tracer

The `langchain.py` module defines the LangChain tracer that records Runnable, model, chain, tool, and retriever runs to LangSmith.

It manages LangSmith client selection, project and example association, inherited tags and metadata, root-run URLs, runtime-environment metadata, run creation through `POST`, run completion through `PATCH`, and client flushing.

## Constants

1. `OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS`: Stores LangSmith-only metadata keys that may be replaced by the nearest nested tracer configuration.

   Most tracing metadata follows a first-value-wins rule so child configurations cannot overwrite values inherited from an ancestor. Keys in this allowlist bypass that rule.

   * **Definition:**
     ```python
     OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS: frozenset[str] = frozenset(
         {"ls_agent_type"}
     )
     ```

### Functions

1. `log_error_once`: Logs an exception only once for each combination of method name and exception type.

   Repeated exceptions of the same type from the same method are ignored, even when their messages differ.

   * **Syntax:**
     ```python
     log_error_once(
         method: str, # Name of the method that raised the exception
         exception: Exception # Exception to log
     ) -> None
     ```

2. `wait_for_all_tracers`: Flushes the globally cached LangSmith client when it has already been created.

   This waits for pending tracing operations managed by that client to finish.

   * **Syntax:**
     ```python
     wait_for_all_tracers() -> None
     ```

3. `get_client`: Returns the cached global LangSmith client.
   * **Syntax:**
     ```python
     get_client() -> Client
     ```

# LangChainTracer

`LangChainTracer` is a synchronous tracer that sends LangChain runs to LangSmith.

Runs are associated with a LangSmith project, may be linked to a reference example, and inherit tracer-level tags and metadata. Start callbacks generally create runs through `POST`, while successful and failed completion callbacks update existing runs through `PATCH`.

## Bases

- `BaseTracer`

## Attributes

1. `run_inline`: Indicates that callback processing should execute inline.
   * **Type:**
     ```python
     run_inline: bool = True
     ```

2. `example_id`: Stores the optional LangSmith example associated with root runs.

   String values supplied to the constructor are converted to `UUID` objects.

   * **Type:**
     ```python
     example_id: UUID | None
     ```

3. `project_name`: Stores the LangSmith project or tracing session name.

   When no project is supplied, the configured tracer project is used.

   * **Type:**
     ```python
     project_name: str
     ```

4. `client`: Stores the LangSmith client used to create, update, flush, and locate runs.
   * **Type:**
     ```python
     client: Client
     ```

5. `tags`: Stores tags applied to traced runs in addition to run-specific tags.
   * **Type:**
     ```python
     tags: list[str]
     ```

6. `latest_run`: Stores a reduced copy of the most recently completed root run.

   Child-run trees are excluded to reduce retained memory. This run is used by `get_run_url`.

   * **Type:**
     ```python
     latest_run: Run | None
     ```

7. `run_has_token_event_map`: Tracks whether an LLM or chat-model run has already stored a token event.

   Only the first token event is added to the persisted run representation.

   * **Type:**
     ```python
     run_has_token_event_map: dict[
         str,
         bool
     ]
     ```

8. `tracing_metadata`: Stores tracer-level metadata defaults.

   Missing run metadata keys are filled from this mapping before a run is posted. Allowlisted LangSmith-only keys may overwrite inherited values.

   * **Type:**
     ```python
     tracing_metadata: dict[
         str,
         str
     ] | None
     ```

### Methods

1. `__init__`: Initializes the LangSmith tracer.

   The constructor resolves the reference example, project, client, tags, and tracer-level metadata. Additional arguments are passed to `BaseTracer`, allowing run and ordering maps to be shared with copied tracer instances.

   * **Syntax:**
     ```python
     __init__(
         self,
         example_id: UUID | str | None = None, # Optional reference-example identifier
         project_name: str | None = None, # Optional LangSmith project name
         client: Client | None = None, # Optional LangSmith client
         tags: list[str] | None = None, # Tags applied to traced runs
         *,
         metadata: Mapping[
             str,
             str
         ] | None = None, # Tracer-level metadata defaults
         **kwargs: Any # Additional BaseTracer arguments
     ) -> None
     ```

2. `copy_with_metadata_defaults`: Creates a new tracer that shares run bookkeeping while applying merged metadata and tag defaults.

   Existing metadata normally wins over newly supplied metadata. Keys in `OVERRIDABLE_LANGSMITH_INHERITABLE_METADATA_KEYS` may instead be replaced by the new value. Newly supplied tags are combined with existing tags, deduplicated, and sorted.

   The copied tracer preserves the example, project, client, run map, ordering map, and external-parent reference counts.

   * **Syntax:**
     ```python
     copy_with_metadata_defaults(
         self,
         *,
         metadata: Mapping[
             str,
             str
         ] | None = None, # Additional tracer metadata defaults
         tags: list[str] | None = None # Additional tracer tags
     ) -> LangChainTracer
     ```

3. `_start_trace`: Applies tracer-level project, tags, client, and tracing-state information before registering a run.

   The project becomes the run's session name. Run-specific and tracer-level tags are merged and deduplicated. The tracer client is attached when the run has no client.

   When the active LangSmith tracing context is explicitly disabled, the run is marked so later persistence and update operations are skipped.

   * **Syntax:**
     ```python
     _start_trace(
         self,
         run: Run # Run to initialize and register
     ) -> None
     ```

4. `on_chat_model_start`: Creates and starts a chat-model run.

   Input messages are serialized with `dumpd`. Metadata is inserted into the run's extra values, a start event is created with a UTC timestamp, and the run is registered and persisted through the chat-model start hook.

   The stored run type is `"llm"` for compatibility with LangSmith model tracing.

   * **Syntax:**
     ```python
     on_chat_model_start(
         self,
         serialized: dict[
             str,
             Any
         ], # Serialized chat model
         messages: list[
             list[BaseMessage]
         ], # Batches of input messages
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[
             str,
             Any
         ] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run properties
     ) -> Run
     ```

5. `_persist_run`: Stores a reduced copy of a completed root run as `latest_run`.

   The copied run excludes `child_runs` while preserving its own inputs and outputs. This method does not send the run to LangSmith; network persistence is handled by `_persist_run_single`.

   * **Syntax:**
     ```python
     _persist_run(
         self,
         run: Run # Completed root run
     ) -> None
     ```

6. `get_run_url`: Returns the LangSmith URL of `latest_run`.

   The client lookup is retried up to five times for `LangSmithError` failures using exponential backoff with jitter. A `ValueError` is raised when no completed root run is available or no URL can be obtained.

   * **Syntax:**
     ```python
     get_run_url(
         self
     ) -> str
     ```

7. `_get_tags`: Combines a run's tags with the tracer-level tags.

   Duplicate values are removed.

   * **Syntax:**
     ```python
     _get_tags(
         self,
         run: Run # Run whose tags are merged
     ) -> list[str]
     ```

8. `_persist_run_single`: Creates one run in LangSmith.

   Disabled runs are ignored. Before posting, the method adds runtime-environment information, merges tags, fills missing tracer metadata, and ensures the run uses the tracer's client.

   Persistence errors are logged through `log_error_once` and then re-raised.

   * **Syntax:**
     ```python
     _persist_run_single(
         self,
         run: Run # Run to post to LangSmith
     ) -> None
     ```

9. `_update_run_single`: Updates one existing LangSmith run.

   Disabled runs are ignored. Inputs are excluded from the patch when the run's extra values contain a truthy `inputs_is_truthy` marker.

   Update errors are logged through `log_error_once` and then re-raised.

   * **Syntax:**
     ```python
     @staticmethod
     _update_run_single(
         run: Run # Run to patch in LangSmith
     ) -> None
     ```

10. `_on_llm_start`: Persists a newly started LLM run.

    Root runs are associated with `example_id` before being posted.

    * **Syntax:**
      ```python
      _on_llm_start(
          self,
          run: Run # Started LLM run
      ) -> None
      ```

11. `_llm_run_with_token_event`: Adds only the first token event received for an LLM or chat-model run.

    Later token callbacks return the active run without appending another token event. Generation chunks are intentionally discarded from the stored event.

    * **Syntax:**
      ```python
      _llm_run_with_token_event(
          self,
          token: str
          | list[
              str
              | dict[
                  str,
                  Any
              ]
          ], # Token or structured content blocks
          run_id: UUID, # Run receiving the token
          chunk: GenerationChunk
          | ChatGenerationChunk
          | None = None, # Generation chunk that is not persisted
          parent_run_id: UUID | None = None # Optional parent-run identifier
      ) -> Run
      ```

12. `_on_chat_model_start`: Persists a newly started chat-model run.

    Root runs are associated with `example_id` before being posted. Chat-model completion is handled through `_on_llm_end`.

    * **Syntax:**
      ```python
      _on_chat_model_start(
          self,
          run: Run # Started chat-model run
      ) -> None
      ```

13. `_on_llm_end`: Updates a successfully completed LLM or chat-model run.

    Usage metadata found in serialized generation messages is aggregated and stored under `run.extra["metadata"]["usage_metadata"]` before the run is patched.

    * **Syntax:**
      ```python
      _on_llm_end(
          self,
          run: Run # Completed LLM or chat-model run
      ) -> None
      ```

14. `_on_llm_error`: Patches an LLM or chat-model run after an error.
    * **Syntax:**
      ```python
      _on_llm_error(
          self,
          run: Run # Errored LLM or chat-model run
      ) -> None
      ```

15. `_on_chain_start`: Persists a newly started chain run.

    Root runs are associated with `example_id`. Runs whose inputs are deferred are not posted until their inputs have been realized at completion.

    * **Syntax:**
      ```python
      _on_chain_start(
          self,
          run: Run # Started chain run
      ) -> None
      ```

16. `_on_chain_end`: Finalizes a successfully completed chain run.

    A run with deferred inputs is posted for the first time after those inputs are realized. Other chain runs are patched.

    * **Syntax:**
      ```python
      _on_chain_end(
          self,
          run: Run # Completed chain run
      ) -> None
      ```

17. `_on_chain_error`: Finalizes an errored chain run.

    A run with deferred inputs is posted after the error when its inputs are available. Other chain runs are patched.

    * **Syntax:**
      ```python
      _on_chain_error(
          self,
          run: Run # Errored chain run
      ) -> None
      ```

18. `_on_tool_start`: Persists a newly started tool run.

    Root runs are associated with `example_id` before being posted.

    * **Syntax:**
      ```python
      _on_tool_start(
          self,
          run: Run # Started tool run
      ) -> None
      ```

19. `_on_tool_end`: Patches a successfully completed tool run.
    * **Syntax:**
      ```python
      _on_tool_end(
          self,
          run: Run # Completed tool run
      ) -> None
      ```

20. `_on_tool_error`: Patches a tool run after an error.
    * **Syntax:**
      ```python
      _on_tool_error(
          self,
          run: Run # Errored tool run
      ) -> None
      ```

21. `_on_retriever_start`: Persists a newly started retriever run.

    Root runs are associated with `example_id` before being posted.

    * **Syntax:**
      ```python
      _on_retriever_start(
          self,
          run: Run # Started retriever run
      ) -> None
      ```

22. `_on_retriever_end`: Patches a successfully completed retriever run.
    * **Syntax:**
      ```python
      _on_retriever_end(
          self,
          run: Run # Completed retriever run
      ) -> None
      ```

23. `_on_retriever_error`: Patches a retriever run after an error.
    * **Syntax:**
      ```python
      _on_retriever_error(
          self,
          run: Run # Errored retriever run
      ) -> None
      ```

24. `wait_for_futures`: Flushes the tracer's LangSmith client.

    This waits for pending client-side tracing operations to complete.

    * **Syntax:**
      ```python
      wait_for_futures(
          self
      ) -> None
      ```